## Silver Layer Transformation - Merit Order Data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime

# Configuration
CATALOG = "rf_assessment"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
SOURCE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.merit_order_data"
TARGET_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.merit_order_data"

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_TABLE}")

In [0]:
# Load bronze data
bronze_df = spark.table(SOURCE_TABLE)

print(f"Total records in bronze: {bronze_df.count():,}")
print(f"\nSchema:")
bronze_df.printSchema()

## Data Quality Checks

In [0]:
# Check for null values in critical columns
print("=" * 80)
print("NULL VALUE ANALYSIS")
print("=" * 80)

null_counts = bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in bronze_df.columns
]).collect()[0].asDict()

print(f"\nTotal records: {bronze_df.count():,}\n")
for col, null_count in null_counts.items():
    if null_count > 0:
        pct = (null_count / bronze_df.count()) * 100
        print(f"  {col:25s}: {null_count:6,} nulls ({pct:5.2f}%)")

if all(count == 0 for count in null_counts.values()):
    print("  ✓ No null values found in any column")

### Check 1: Null Values in Critical Columns

In [0]:
# Check for duplicate records
print("=" * 80)
print("DUPLICATE DETECTION")
print("=" * 80)

# Note: Same plant can appear multiple times with DIFFERENT fuel types
# True duplicates = identical records (all key fields same)
duplicate_check = bronze_df.groupBy(
    "plant_name", "effective_date", "file_name", "fuel_type",
    "other_cost", "fuel_cost", "vom_cost", "specific_cost"
).count().filter(F.col("count") > 1)

dup_count = duplicate_check.count()

if dup_count > 0:
    print(f"\n  ⚠️  Found {dup_count} true duplicate records (identical values)")
    print("\nSample duplicates:")
    display(duplicate_check.orderBy(F.desc("count")).limit(10))
else:
    print("\n  ✓ No duplicate records found")
    print("  Note: Same plant with different fuel types is valid data, not a duplicate")

### Check 2: Duplicate Records Detection

In [0]:
# Validate data ranges and business logic
print("=" * 80)
print("DATA VALIDATION")
print("=" * 80)

# Check for negative cost values (should not exist)
negative_costs = bronze_df.filter(
    (F.col("other_cost") < 0) | 
    (F.col("fuel_cost") < 0) | 
    (F.col("vom_cost") < 0) | 
    (F.col("specific_cost") < 0)
).count()

print(f"\n1. Negative cost values: {negative_costs}")
if negative_costs > 0:
    print("   ⚠️  Warning: Found negative cost values")
else:
    print("   ✓ All cost values are non-negative")

# Check date format and validity
invalid_dates = bronze_df.filter(
    F.col("effective_date").rlike(r'^\d{4}-\d{2}-\d{2}$') == False
).count()

print(f"\n2. Invalid date format: {invalid_dates}")
if invalid_dates > 0:
    print("   ⚠️  Warning: Found dates not in YYYY-MM-DD format")
else:
    print("   ✓ All dates are in correct format")

# Check year and month consistency
print(f"\n3. Date consistency check:")
date_consistency = bronze_df.withColumn(
    "date_parsed", F.to_date(F.col("effective_date"))
).withColumn(
    "year_from_date", F.year("date_parsed")
).withColumn(
    "month_from_date", F.month("date_parsed")
).filter(
    (F.col("year") != F.col("year_from_date")) | 
    (F.col("month") != F.col("month_from_date"))
).count()

if date_consistency > 0:
    print(f"   ⚠️  Warning: {date_consistency} records with year/month mismatch")
else:
    print("   ✓ Year and month columns match effective_date")

# Check fuel type distribution
print("\n4. Fuel type distribution:")
fuel_dist = bronze_df.groupBy("fuel_type").count().orderBy(F.desc("count"))
display(fuel_dist)

### Check 3: Data Ranges and Business Logic Validation

## Silver Layer Transformations

Applying data quality improvements and enrichments to prepare data for gold layer analysis.

In [0]:
print("=" * 80)
print("APPLYING SILVER TRANSFORMATIONS")
print("=" * 80)

# Initialize silver dataframe from bronze
silver_df = bronze_df
print(f"\nStarting with {silver_df.count():,} records from bronze layer")

### 1. Convert Data Types

In [0]:
# Convert effective_date from string to proper date type
silver_df = silver_df.withColumn(
    "effective_date", 
    F.to_date(F.col("effective_date"))
)

print("✓ Converted effective_date to date type")

### 2. Clean and Standardize Text Fields

In [0]:
# Clean plant names - remove leading/trailing whitespace
silver_df = silver_df.withColumn(
    "plant_name",
    F.trim(F.col("plant_name"))
)

print("✓ Cleaned plant names (trimmed whitespace)")

In [0]:
# Standardize fuel types - uppercase and trim for consistency
silver_df = silver_df.withColumn(
    "fuel_type_clean",
    F.upper(F.trim(F.col("fuel_type")))
)

print("✓ Standardized fuel types (uppercase)")

### 3. Add Data Quality Flags

In [0]:
# Add flag to identify records with valid (non-negative) cost values
silver_df = silver_df.withColumn(
    "has_valid_costs",
    (F.col("other_cost") >= 0) & 
    (F.col("fuel_cost") >= 0) & 
    (F.col("vom_cost") >= 0) & 
    (F.col("specific_cost") >= 0)
)

print("✓ Added data quality flags")

### 4. Calculate Cost Breakdown Percentages

In [0]:
# Calculate what percentage of specific_cost comes from each component
silver_df = silver_df.withColumn(
    "fuel_cost_pct",
    F.when(F.col("specific_cost") > 0, 
           (F.col("fuel_cost") / F.col("specific_cost") * 100)
    ).otherwise(0)
).withColumn(
    "vom_cost_pct",
    F.when(F.col("specific_cost") > 0,
           (F.col("vom_cost") / F.col("specific_cost") * 100)
    ).otherwise(0)
).withColumn(
    "other_cost_pct",
    F.when(F.col("specific_cost") > 0,
           (F.col("other_cost") / F.col("specific_cost") * 100)
    ).otherwise(0)
)

print("✓ Calculated cost breakdown percentages")

### 5. Select Essential Columns for Gold Layer

Optimized for **plant-wise fuel cost variation analysis**:
* Plant & fuel identifiers (plant_name, fuel_type_clean)
* Time dimensions (effective_date, month, year, file_name)
* Cost metrics (fuel_cost, specific_cost, other_cost, vom_cost)
* Percentage breakdowns (fuel_cost_pct, vom_cost_pct, other_cost_pct)

**Removed columns:** sr_no, fuel_type (raw), status_last_order, ingestion_date, has_valid_costs

In [0]:
# Select only the columns needed for gold layer analysis
silver_df = silver_df.select(
    "plant_name",
    "fuel_type_clean",
    "effective_date",
    "month",
    "year",
    "fuel_cost",
    "specific_cost",
    "other_cost",
    "vom_cost",
    "fuel_cost_pct",
    "vom_cost_pct",
    "other_cost_pct",
    "file_name"
)

print("✓ Selected 13 essential columns for gold layer")
print(f"\nTotal records: {silver_df.count():,}")
print(f"Columns: {len(silver_df.columns)}")

### Preview Transformed Data

In [0]:
# Preview transformed data
print("\nSilver DataFrame Schema:")
silver_df.printSchema()

print("\nSample records:")
display(silver_df.orderBy("effective_date", "plant_name").limit(10))

## Write to Silver Delta Table

In [0]:
# Ensure silver schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
print(f"✓ Schema {CATALOG}.{SILVER_SCHEMA} is ready")

### Prepare Silver Schema

In [0]:
print("=" * 80)
print("WRITING TO SILVER DELTA TABLE")
print("=" * 80)

# Write to silver table with partitioning for efficient time-series reads
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("month", "file_name") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE)

print(f"\n✓ Successfully wrote {silver_df.count():,} records to {TARGET_TABLE}")
print(f"Partitioned by: month, file_name")

### Write Data with Partitioning

### Optimize Table for Query Performance

In [0]:
# Apply Z-ORDER by plant_name for faster plant-wise filtering in gold layer
print("Optimizing table with Z-ORDER by plant_name...")
spark.sql(f"OPTIMIZE {TARGET_TABLE} ZORDER BY (plant_name)")
print("✓ Table optimized for plant-wise queries")

### Display Table Statistics

In [0]:
# Show final table statistics
final_count = spark.table(TARGET_TABLE).count()

print("=" * 80)
print("SILVER TABLE STATISTICS")
print("=" * 80)
print(f"Total records: {final_count:,}")
print(f"Columns: {len(spark.table(TARGET_TABLE).columns)}")
print(f"Table location: {TARGET_TABLE}")
print(f"Partitioned by: month, file_name (time-series optimization)")
print(f"Z-ordered by: plant_name (plant-wise query optimization)")

## Post-Write Validation

Verify the silver table was created correctly with expected data distribution.

In [0]:
# Verify silver table was created successfully
silver_table = spark.table(TARGET_TABLE)

print("Records by year and month:")
monthly_dist = silver_table.groupBy("year", "month") \
    .count() \
    .orderBy("year", "month")
display(monthly_dist)

In [0]:
# Count unique plants in the dataset
plant_count = silver_table.select("plant_name").distinct().count()
print(f"Unique plants in dataset: {plant_count}")

In [0]:
# Check fuel type distribution after standardization
print("Fuel type distribution (cleaned):")
fuel_dist_silver = silver_table.groupBy("fuel_type_clean") \
    .count() \
    .orderBy(F.desc("count"))
display(fuel_dist_silver)

In [0]:
print("=" * 80)
print("✓ SILVER LAYER TRANSFORMATION COMPLETED SUCCESSFULLY")
print("=" * 80)

In [0]:
%sql
DESCRIBE DETAIL rf_assessment.silver.merit_order_data;

In [0]:
%sql
SELECT * FROM rf_assessment.bronze.merit_order_data LIMIT 10;

In [0]:
%sql
SELECT * FROM rf_assessment.silver.merit_order_data LIMIT 10;